In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load in 

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the "../input/" directory.
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory


# Any results you write to the current directory are saved as output.

import datetime
import pandas as pd
import os
import numpy as np
from matplotlib import pyplot
from math import sqrt
import matplotlib.pyplot as plt
%matplotlib inline
import os
import seaborn as sns
import statsmodels.api as sm
import warnings

from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.stattools import adfuller, kpss, ccf
from statsmodels.tsa.ar_model import AutoReg
from statsmodels.tsa.arima.model import ARIMA 

from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import LabelEncoder, StandardScaler, MinMaxScaler
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from numpy import array
import torch
import gc
import torch.nn as nn
#from tqdm import tqdm_notebook as tqdm
from torch.utils.data import Dataset,DataLoader

In [3]:
os.chdir('C:\\Users\\loren\\OneDrive - Università degli Studi di Milano\\Lezioni uni\\Tesi\\Dataset Energia\\')
path = 'Italy'

In [4]:
df2016 = pd.read_csv(os.path.join(path, f"ITA2016.csv"), parse_dates = ['MTU'])
df2017 = pd.read_csv(os.path.join(path, f"ITA2017.csv"), parse_dates = ['MTU'])
df2018 = pd.read_csv(os.path.join(path, f"ITA2018.csv"), parse_dates = ['MTU'])
df2019 = pd.read_csv(os.path.join(path, f"ITA2019.csv"), parse_dates = ['MTU'])
df2020 = pd.read_csv(os.path.join(path, f"ITA2020.csv"), parse_dates = ['MTU'])
df2021 = pd.read_csv(os.path.join(path, f"ITA2021.csv"), parse_dates = ['MTU'])

In [5]:
def retrieve_data(path):
    # creo il vettore degli anni e il df vuoto dove appendere i singoli df
    years = [2016, 2017, 2018, 2019, 2020, 2021]
    tot = pd.DataFrame()
    
    # carico i dati dei diversi anni
    for year in years:
        df = pd.read_csv(os.path.join(path, f"ITA{year}.csv"), parse_dates = ['MTU'])
        #df = df.reset_index()
        
        # aggiusto il formato della data in YYYY-MM-DD HH:mm
        for i, row in df.iterrows():
            df["MTU"][i] = df['MTU'][i][:16]
        
        df['Time'] = pd.to_datetime(df['MTU'], utc=True, infer_datetime_format=True)
        df.Time = df.Time.dt.strftime('%Y-%m-%d')
        df = df.set_index(pd.DatetimeIndex(df['Time']))

        # elimino colonne con solo NA
        df = df.drop(columns=['MTU','Area','Fossil Brown coal/Lignite  - Actual Aggregated [MW]','Fossil Oil shale  - Actual Aggregated [MW]',
                     'Fossil Peat  - Actual Aggregated [MW]', 'Marine  - Actual Aggregated [MW]', 'Nuclear  - Actual Aggregated [MW]',
                     'Wind Offshore  - Actual Aggregated [MW]', 'Other renewable  - Actual Aggregated [MW]'])
        
        # appendo i vari df a quello vuoto principale
        tot = pd.concat([tot, df], ignore_index=True)
    
    # imposto la data come indice
    tot = tot.set_index(pd.DatetimeIndex(tot['Time']))
    return tot

In [6]:
data = retrieve_data(path)

C:\Users\loren\AppData\Local\Temp\ipykernel_21308\2564700696.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["MTU"][i] = df['MTU'][i][:16]
C:\Users\loren\AppData\Local\Temp\ipykernel_21308\2564700696.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["MTU"][i] = df['MTU'][i][:16]
C:\Users\loren\AppData\Local\Temp\ipykernel_21308\2564700696.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["MTU"][i] = df['MTU'][i][:16

In [7]:
data.interpolate(method='linear', limit_direction='forward', inplace=True, axis=0)

In [8]:
data.describe()

,Biomass - Actual Aggregated [MW],Fossil Coal-derived gas - Actual Aggregated [MW],Fossil Gas - Actual Aggregated [MW],Fossil Hard coal - Actual Aggregated [MW],Fossil Oil - Actual Aggregated [MW],Geothermal - Actual Aggregated [MW],Hydro Pumped Storage - Actual Aggregated [MW],Hydro Pumped Storage - Actual Consumption [MW],Hydro Run-of-river and poundage - Actual Aggregated [MW],Hydro Water Reservoir - Actual Aggregated [MW],Other - Actual Aggregated [MW],Solar - Actual Aggregated [MW],Waste - Actual Aggregated [MW],Wind Onshore - Actual Aggregated [MW]
count,52614.000000,52614.000000,52614.000000,52614.000000,52614.000000,52614.000000,52614.000000,52614.000000,52614.000000,52614.000000,52614.000000,52614.000000,52614.000000,52614.000000
mean,491.304482,411.358526,10398.377352,1876.884736,135.662457,650.406812,380.245800,285.435816,3818.574676,789.196203,5584.579665,2185.143498,37.795957,2114.967784
std,165.546821,241.377903,4702.358879,967.511420,116.660972,22.489450,518.017544,422.781559,1577.662605,489.626447,3856.001692,3040.677333,8.574162,1525.440236
min,167.000000,0.000000,1590.000000,204.000000,0.000000,485.000000,0.000000,1.000000,774.000000,14.000000,623.000000,0.000000,5.000000,20.000000
25%,334.000000,232.000000,6674.000000,1170.000000,39.000000,637.000000,5.000000,25.000000,2540.000000,402.000000,2222.000000,0.000000,32.000000,858.000000
50%,481.000000,280.000000,9641.000000,1582.000000,93.000000,653.000000,167.000000,112.750000,3691.000000,708.000000,4079.500000,60.000000,39.000000,1753.000000
75%,652.000000,592.000000,13422.750000,2452.000000,209.000000,667.000000,542.000000,349.000000,4975.000000,1115.000000,8445.750000,4178.000000,44.000000,3084.000000
max,845.000000,1135.000000,27688.000000,6198.000000,783.000000,707.000000,4526.000000,3974.000000,8534.000000,2872.000000,19489.000000,11812.000000,62.000000,7611.000000


In [9]:
def daily_data_and_aggregator(df):

    # seleziono tutte le colonne delle fonti energetiche
    columns = df.columns[ : data.shape[1]-1]

    # aggrego giornalmente i dati
    daily_df = pd.DataFrame(df[columns].groupby([pd.Grouper(level='Time', freq='D')]).sum())

    # creo l'aggregato totale
    daily_df['total_aggregated'] = daily_df[columns].sum(axis=1)

    return daily_df

In [10]:
daily_data = daily_data_and_aggregator(data)

In [11]:
daily_data.total_aggregated.describe()

count      2192.000000
mean     699918.227646
std      103382.026351
min      459101.000000
25%      614134.500000
50%      708275.986111
75%      776133.742424
max      978987.375000
Name: total_aggregated, dtype: float64

In [12]:
# creo due dunzioni: una per i weekend (quando sabato/domenica è 1, altrimenti 0) e l'altra per le vacanze (1/1, 25/4, 1/5, 2/6, 15/8, 25/12)
def weekend_generation(df):
    # Generate 'weekend' feature
    for i in range(len(df)):
        position = df.index[i]
        hour = position.hour
        weekend = position.weekday()
        month = position.month
        df.loc[position, 'weekend'] = weekend

    for i in range(len(df)):
        position = df.index[i]
        weekend = position.weekday()
        if (weekend >= 5):
            df.loc[position, 'weekend'] = 1
        else:
            df.loc[position, 'weekend'] = 0
    return df

In [13]:
daily_data2 = weekend_generation(daily_data)

In [14]:
def create_sliding_window(data, sequence_length, stride=1):
    X_list, y_list = [], []
    for i in range(len(data)):
      if (i + sequence_length) < len(data):
        X_list.append(data.iloc[i:i+sequence_length:stride, :].values)
        y_list.append(data.iloc[i+sequence_length, -1])
    return np.array(X_list), np.array(y_list)

train_split = 0.7
n_train = 1460
n_test = 732

features = ['total_aggregated']
feature_array = daily_data2[features].values

# Fit Scaler only on Training features
feature_scaler = MinMaxScaler()
feature_scaler.fit(feature_array[:n_train])
# Fit Scaler only on Training target values
target_scaler = MinMaxScaler(feature_range = (-1,1))
target_scaler.fit(feature_array[:n_train, -1].reshape(-1, 1))

# Transfom on both Training and Test data
scaled_array = pd.DataFrame(feature_scaler.transform(feature_array),
                            columns=features)

sequence_length = 1
X, y = create_sliding_window(scaled_array, 
                             sequence_length)

X_train = X[:n_train]
y_train = y[:n_train]

X_test = X[n_train:]
y_test = y[n_train:]

In [15]:
class CNN_ForecastNet(nn.Module):
    def __init__(self):
        super(CNN_ForecastNet,self).__init__()
        self.conv1d = nn.Conv1d(3,64,kernel_size=1)
        self.relu = nn.ReLU(inplace=True)
        self.fc1 = nn.Linear(64*2,50)
        self.fc2 = nn.Linear(50,1)
        
    def forward(self,x):
        x = self.conv1d(x)
        x = self.relu(x)
        x = x.view(-1)
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        
        return x

In [16]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model = CNN_ForecastNet()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-5)
criterion = nn.MSELoss()

In [ ]:
train = daily_data2(X_train.reshape(X_train.shape[0],X_train.shape[1],1),y_train)
valid = daily_data2(X_test.reshape(X_test.shape[0],X_test.shape[1],1),y_test)
train_loader = torch.utils.data.DataLoader(train,batch_size=2,shuffle=False)
valid_loader = torch.utils.data.DataLoader(train,batch_size=2,shuffle=False)

In [ ]:
train_losses = []
valid_losses = []
def Train():
    
    running_loss = .0
    
    model.train()
    
    for idx, (inputs,labels) in enumerate(train_loader):
        inputs = inputs.to(device)
        labels = labels.to(device)
        optimizer.zero_grad()
        preds = model(inputs.float())
        loss = criterion(preds,labels)
        loss.backward()
        optimizer.step()
        running_loss += loss
        
    train_loss = running_loss/len(train_loader)
    train_losses.append(train_loss.detach().numpy())
    
    print(f'train_loss {train_loss}')
    
def Valid():
    running_loss = .0
    
    model.eval()
    
    with torch.no_grad():
        for idx, (inputs, labels) in enumerate(valid_loader):
            inputs = inputs.to(device)
            labels = labels.to(device)
            optimizer.zero_grad()
            preds = model(inputs.float())
            loss = criterion(preds,labels)
            running_loss += loss
            
        valid_loss = running_loss/len(valid_loader)
        valid_losses.append(valid_loss.detach().numpy())
        print(f'valid_loss {valid_loss}')

In [ ]:
epochs = 200
for epoch in range(epochs):
    print('epochs {}/{}'.format(epoch+1,epochs))
    Train()
    Valid()
    gc.collect()

In [ ]:
target_x , target_y = X_test, y_test
inputs = target_x.reshape(target_x.shape[0],target_x.shape[1],1)

In [ ]:
model.eval()
prediction = []
batch_size = 2
iterations =  int(inputs.shape[0]/2)

for i in range(iterations):
    preds = model(torch.tensor(inputs[batch_size*i:batch_size*(i+1)]).float())
    prediction.append(preds.detach().numpy())

In [ ]:
fig, ax = plt.subplots(1, 2,figsize=(11,4))
ax[0].set_title('predicted one')
ax[0].plot(prediction)
ax[1].set_title('real one')
ax[1].plot(target_y)
plt.show()